# Learning Curves — Accuracy & Brier vs Training Set Size

**Prerequisites:** Run `data/01_build_features.ipynb` to generate `data/latest_features.jsonl`.

**Goal:** Train LR, RF, XGB, DNN-features, and DNN-raw on increasing data subsets.
Measure accuracy and Brier score via GroupKFold CV at each size.
Answer: how much data does each model need to reach peak performance?

**Models:**
- **LR, RF, XGB**: Use optimal features + hyperparameters from `data/optimal_features_*.json`
- **DNN-features**: Same architecture as `02_advanced_models`, all 60 engineered features
- **DNN-raw**: Same architecture but 11 raw inputs only (price, orderbook, volume) — can DNN learn features from raw data?

In [ ]:
import sys

sys.path.insert(0, str(__import__("pathlib").Path.cwd().parent))

import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from deep_neural_network import DeepNeuralNetworkRunner
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

try:
    import xgboost as xgb_lib
except ImportError:
    xgb_lib = None

random.seed(42)
np.random.seed(42)

FEATURES_PATH = Path("../../data/latest_features.jsonl")

## 1. Load data and model configs

In [ ]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

NON_FEAT = {
    "candle_id",
    "session",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
}
all_feat_cols = sorted([c for c in df.columns if c not in NON_FEAT])
df[all_feat_cols] = df[all_feat_cols].fillna(0.0)

# Raw columns for DNN-raw (price, orderbook, volume — no engineered features)
RAW_COLS = sorted(
    [
        "btc_price",
        "elapsed_pct",
        "market_volume",
        "up_best_bid",
        "up_best_ask",
        "up_bid_depth",
        "up_ask_depth",
        "down_best_bid",
        "down_best_ask",
        "down_bid_depth",
        "down_ask_depth",
    ]
)

# Load per-model feature configs
model_configs = {}
for name, path in [("LR", "lr"), ("RF", "rf"), ("XGB", "xgb")]:
    with open(Path(f"../../data/optimal_features_{path}.json")) as f:
        cfg = json.load(f)
    model_configs[name] = {
        "features": sorted(cfg["features"]),
        "hyperparams": cfg.get("hyperparameters", {}),
    }

candle_ids = df["candle_id"].unique()
print(f"Total: {len(df):,} rows, {len(candle_ids)} candles, {len(all_feat_cols)} features")
print(f"Raw columns for DNN-raw: {len(RAW_COLS)}")
for name, cfg in model_configs.items():
    print(f"  {name}: {len(cfg['features'])} features, params={cfg['hyperparams']}")

## 2. Learning curve engine

For each model, for each training size N:
- Take the first N candles (time-ordered)
- Run 5-fold GroupKFold CV (group by candle_id)
- Record accuracy and Brier score (mean ± std across folds)

In [ ]:
TRAIN_SIZES = [100, 200, 400, 600, 800, 1000, 1500, 2000, 2500, 3000, 3500, 4000]
TRAIN_SIZES = [s for s in TRAIN_SIZES if s <= len(candle_ids)]
N_FOLDS = 5

print(f"Training sizes: {TRAIN_SIZES}")
print(f"Max available: {len(candle_ids)} candles")


def evaluate_cv(model_fn, feature_cols, df_subset, n_folds=N_FOLDS):
    """Run GroupKFold CV and return (accuracy_list, brier_list) per fold."""
    groups = df_subset["candle_id"].values
    X = df_subset[feature_cols].values
    y = df_subset["target"].values

    gkf = GroupKFold(n_splits=n_folds)
    accs, briers = [], []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        model = model_fn()
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)[:, 1]
        preds = (probs >= 0.5).astype(int)

        accs.append(accuracy_score(y_val, preds))
        briers.append(brier_score_loss(y_val, probs))

    return accs, briers


def evaluate_dnn_cv(feature_cols, df_subset, n_folds=N_FOLDS):
    """Run GroupKFold CV for DNN (subprocess runner)."""
    groups = df_subset["candle_id"].values
    X = df_subset[feature_cols].values
    y = df_subset["target"].values

    gkf = GroupKFold(n_splits=n_folds)
    accs, briers = [], []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        # Use 5% of training data for DNN's internal early-stopping validation
        n_es = max(100, int(len(X_tr) * 0.05))
        X_es, y_es = X_tr[-n_es:], y_tr[-n_es:]
        X_tr_dnn, y_tr_dnn = X_tr[:-n_es], y_tr[:-n_es]

        runner = DeepNeuralNetworkRunner(X_tr_dnn, y_tr_dnn, X_es, y_es)
        runner.setup()
        runner.train(epochs=20, patience=5)
        probs = runner.predict_proba(X_val)
        preds = (probs >= 0.5).astype(int)

        accs.append(accuracy_score(y_val, preds))
        briers.append(brier_score_loss(y_val, probs))

    return accs, briers

## 3. Run learning curves for LR, RF, XGB

In [ ]:
# Model factories
lr_hp = model_configs["LR"]["hyperparams"]
rf_hp = {k: v for k, v in model_configs["RF"]["hyperparams"].items() if k != "n_jobs"}
xgb_hp = model_configs["XGB"]["hyperparams"]
INT_PARAMS = {"max_depth", "min_child_weight", "n_estimators"}
xgb_hp = {k: int(v) if k in INT_PARAMS else float(v) for k, v in xgb_hp.items()}


def make_lr():
    return LogisticRegression(
        C=lr_hp.get("C", 1.0),
        solver=lr_hp.get("solver", "saga"),
        max_iter=lr_hp.get("max_iter", 5000),
        random_state=42,
    )


def make_rf():
    return RandomForestClassifier(n_jobs=-1, **rf_hp)


def make_xgb():
    return xgb_lib.XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
        **xgb_hp,
    )


sklearn_models = {
    "LR": (make_lr, model_configs["LR"]["features"]),
    "RF": (make_rf, model_configs["RF"]["features"]),
    "XGB": (make_xgb, model_configs["XGB"]["features"]),
}

# Run learning curves for sklearn-style models
results = {}
for model_name, (model_fn, feat_cols) in sklearn_models.items():
    print(f"\n{'=' * 50}")
    print(f"  {model_name} ({len(feat_cols)} features)")
    print(f"{'=' * 50}")
    acc_means, acc_stds, brier_means, brier_stds = [], [], [], []

    for n_candles in tqdm(TRAIN_SIZES, desc=model_name):
        subset_ids = set(candle_ids[:n_candles])
        df_sub = df[df["candle_id"].isin(subset_ids)]

        # Need at least N_FOLDS unique candles
        if df_sub["candle_id"].nunique() < N_FOLDS:
            acc_means.append(np.nan)
            acc_stds.append(np.nan)
            brier_means.append(np.nan)
            brier_stds.append(np.nan)
            continue

        accs, briers = evaluate_cv(model_fn, feat_cols, df_sub)
        acc_means.append(np.mean(accs))
        acc_stds.append(np.std(accs))
        brier_means.append(np.mean(briers))
        brier_stds.append(np.std(briers))

        print(
            f"  {n_candles:>5} candles: acc={np.mean(accs) * 100:.1f}% ± {np.std(accs) * 100:.1f}%  brier={np.mean(briers):.4f} ± {np.std(briers):.4f}"
        )

    results[model_name] = {
        "acc_mean": acc_means,
        "acc_std": acc_stds,
        "brier_mean": brier_means,
        "brier_std": brier_stds,
    }

## 4. Run learning curves for DNN-features and DNN-raw

DNN runs in a subprocess (avoids libomp conflict with XGBoost). This section is slower — each point trains a neural network 5 times.

In [ ]:
dnn_variants = {
    "DNN-features": all_feat_cols,
    "DNN-raw": RAW_COLS,
}

for dnn_name, feat_cols in dnn_variants.items():
    print(f"\n{'=' * 50}")
    print(f"  {dnn_name} ({len(feat_cols)} features)")
    print(f"{'=' * 50}")
    acc_means, acc_stds, brier_means, brier_stds = [], [], [], []

    for n_candles in tqdm(TRAIN_SIZES, desc=dnn_name):
        subset_ids = set(candle_ids[:n_candles])
        df_sub = df[df["candle_id"].isin(subset_ids)]

        if df_sub["candle_id"].nunique() < N_FOLDS:
            acc_means.append(np.nan)
            acc_stds.append(np.nan)
            brier_means.append(np.nan)
            brier_stds.append(np.nan)
            continue

        accs, briers = evaluate_dnn_cv(feat_cols, df_sub)
        acc_means.append(np.mean(accs))
        acc_stds.append(np.std(accs))
        brier_means.append(np.mean(briers))
        brier_stds.append(np.std(briers))

        print(
            f"  {n_candles:>5} candles: acc={np.mean(accs) * 100:.1f}% ± {np.std(accs) * 100:.1f}%  brier={np.mean(briers):.4f} ± {np.std(briers):.4f}"
        )

    results[dnn_name] = {
        "acc_mean": acc_means,
        "acc_std": acc_stds,
        "brier_mean": brier_means,
        "brier_std": brier_stds,
    }

print("\nAll models complete.")

## 5. Accuracy learning curves

In [ ]:
MODEL_COLORS = {
    "LR": "#3498db",
    "RF": "#2ecc71",
    "XGB": "#e67e22",
    "DNN-features": "#9b59b6",
    "DNN-raw": "#e74c3c",
}

fig, ax = plt.subplots(figsize=(14, 7))

for model_name, data in results.items():
    means = np.array(data["acc_mean"])
    stds = np.array(data["acc_std"])
    valid = ~np.isnan(means)
    sizes = np.array(TRAIN_SIZES)[valid]
    m = means[valid]
    s = stds[valid]

    color = MODEL_COLORS[model_name]
    ax.plot(sizes, m * 100, "o-", color=color, linewidth=2, markersize=5, label=model_name)
    ax.fill_between(sizes, (m - s) * 100, (m + s) * 100, alpha=0.15, color=color)

ax.axhline(50, color="gray", linestyle="--", alpha=0.4, label="Random (50%)")
ax.set_xlabel("Training Set Size (candles)", fontsize=12)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Learning Curves — Accuracy vs Training Set Size", fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim(0, max(TRAIN_SIZES) * 1.05)
plt.tight_layout()
plt.show()

## 6. Brier score learning curves

Lower Brier = better probability calibration (important for confidence-gated strategies).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for model_name, data in results.items():
    means = np.array(data["brier_mean"])
    stds = np.array(data["brier_std"])
    valid = ~np.isnan(means)
    sizes = np.array(TRAIN_SIZES)[valid]
    m = means[valid]
    s = stds[valid]

    color = MODEL_COLORS[model_name]
    ax.plot(sizes, m, "o-", color=color, linewidth=2, markersize=5, label=model_name)
    ax.fill_between(sizes, m - s, m + s, alpha=0.15, color=color)

ax.axhline(0.25, color="gray", linestyle="--", alpha=0.4, label="Random (0.25)")
ax.set_xlabel("Training Set Size (candles)", fontsize=12)
ax.set_ylabel("Brier Score (lower = better)", fontsize=12)
ax.set_title("Learning Curves — Brier Score vs Training Set Size", fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim(0, max(TRAIN_SIZES) * 1.05)
plt.tight_layout()
plt.show()

## 7. Summary table

In [ ]:
# Summary table + knee point detection
print(f"{'Model':<16} {'Size':>6} {'Accuracy (μ±σ)':>18} {'Brier (μ±σ)':>18}")
print("=" * 62)

for model_name in results:
    data = results[model_name]
    for i, n in enumerate(TRAIN_SIZES):
        am, astd = data["acc_mean"][i], data["acc_std"][i]
        bm, bstd = data["brier_mean"][i], data["brier_std"][i]
        if np.isnan(am):
            continue
        print(f"{model_name:<16} {n:>6} {am * 100:>6.1f}% ± {astd * 100:<5.1f}% {bm:>7.4f} ± {bstd:<6.4f}")
    print()

# Knee point: smallest size where accuracy is within 0.5% of the model's best
print("\nKnee Points (within 0.5% of best accuracy):")
print("-" * 50)
for model_name, data in results.items():
    means = np.array(data["acc_mean"])
    valid = ~np.isnan(means)
    if not valid.any():
        continue
    valid_means = means[valid]
    valid_sizes = np.array(TRAIN_SIZES)[valid]
    best = valid_means.max()
    for size, acc in zip(valid_sizes, valid_means, strict=False):
        if acc >= best - 0.005:
            print(f"  {model_name:<16} {size:>5} candles  (acc={acc * 100:.1f}%, best={best * 100:.1f}%)")
            break

## 8. Conclusions

Observations from the learning curves above:

- **Knee point per model** — the smallest training set where each model reaches within 0.5% of its best accuracy. Models below their knee point benefit significantly from more data; models above it have plateaued.
- **DNN-features vs DNN-raw** — shows whether the DNN can learn useful representations from raw inputs, or whether hand-engineered indicators are still necessary at current data sizes.
- **Brier score trends** — if a model's accuracy plateaus but Brier keeps improving, it means probability calibration benefits from more data even when classification doesn't. This directly impacts confidence-gated strategies (like XGB's `conf>0.6` filter).

*Fill in specific findings after running the notebook.*